In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 

In [2]:
"""
================================================================
Train: Experiment — Val gộp 2 lần vào train (để test)
================================================================
Model   : YOLOv8n (from scratch, pretrained=False)
Images  : gốc (không gamma)
Labels  : v11
Val     : KHÔNG dùng val riêng — gộp train + val vào train
          Val được gộp 2 lần để test xem có ra kết quả cũ không
Epochs  : 100
Batch   : 64
IMGSZ   : 640
================================================================
"""
import csv
import gc
import os
import shutil
import random
import sys
import yaml
import logging
from pathlib import Path

import numpy as np
import torch

# ================================================================
# CONFIG
# ================================================================

COMPETITION_ROOT = Path('/kaggle/input/competitions/3-lc-multi-vehicle-detection-challenge/competition_starter')
LABEL_DIR        = Path('/kaggle/input/datasets/cdanht2/cleaned-data-v11/cleaned_data_v11/labels/train')
WORK_DIR = Path('/kaggle/working/runs_V11_val5x_org_200ep')

# ================================================================
# DERIVED PATHS
# ================================================================

DATA_DIR  = COMPETITION_ROOT / 'data'
TRAIN_SRC = DATA_DIR / 'train' / 'images'
VAL_SRC   = DATA_DIR / 'val'   / 'images'
VAL_LBL   = DATA_DIR / 'val'   / 'labels'

# ================================================================
# HYPERPARAMS
# ================================================================

EPOCHS      = 250
BATCH       = 64
IMGSZ       = 640
SEED        = 42
VAL_REPEAT  = 5      # ← số lần gộp val vào train (1 = bình thường, 2 = 2 lần, ...)
RUN_NAME    = f'v11_val{VAL_REPEAT}x_200ep'

# ================================================================
# SETUP
# ================================================================

def setup():
    try:
        from ultralytics import YOLO
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

    import torch
    log.info(f'PyTorch : {torch.__version__}')
    log.info(f'CUDA    : {torch.cuda.is_available()}')
    log.info(f'GPUs    : {torch.cuda.device_count()}')

    if torch.cuda.device_count() >= 2:
        device = [0, 1]
    elif torch.cuda.is_available():
        device = 0
    else:
        device = 'cpu'
    log.info(f'Device  : {device}')

    missing = False
    for p, name in [
        (TRAIN_SRC, 'Train images'),
        (VAL_SRC,   'Val images'),
        (VAL_LBL,   'Val labels'),
        (LABEL_DIR, 'Label dir (train)'),
    ]:
        ok = p.exists()
        log.info(f'  {"✅" if ok else "❌ MISSING"} {name}: {p}')
        if not ok:
            missing = True

    if missing:
        raise FileNotFoundError('❌ Một số path bị thiếu — kiểm tra CONFIG')

    return device


# ================================================================
# SETUP TRAIN DIRECTORY — val gộp 2 lần
# ================================================================

def setup_train_dir() -> None:
    train_dir = WORK_DIR / 'train_data'
    img_dir   = train_dir / 'images'
    lbl_dir   = train_dir / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    # --- 1. Symlink ảnh train + labels train ---
    count_train = 0
    for img in TRAIN_SRC.iterdir():
        if img.suffix.lower() in exts:
            link = img_dir / img.name
            if link.exists() or link.is_symlink():
                link.unlink()
            link.symlink_to(img.resolve())

            lbl_src = LABEL_DIR / (img.stem + '.txt')
            lbl_lnk = lbl_dir  / (img.stem + '.txt')
            if lbl_src.exists():
                if lbl_lnk.exists() or lbl_lnk.is_symlink():
                    lbl_lnk.unlink()
                lbl_lnk.symlink_to(lbl_src.resolve())

            count_train += 1
    log.info(f'  ✅ Train images → {count_train} symlinks')

    # --- 2. Symlink val VAL_REPEAT lần ---
    count_val_total = 0
    for i in range(VAL_REPEAT):
        prefix = '' if i == 0 else f'valx{i}_'   # lần 1: tên gốc, lần 2+: valx1_, valx2_, ...
        count_this = 0
        for img in VAL_SRC.iterdir():
            if img.suffix.lower() not in exts:
                continue
            safe_name = f'{prefix}{img.name}'
            stem      = f'{prefix}{img.stem}'

            link = img_dir / safe_name
            if link.exists() or link.is_symlink():
                link.unlink()
            link.symlink_to(img.resolve())

            lbl_src = VAL_LBL / (img.stem + '.txt')
            lbl_lnk = lbl_dir / (stem + '.txt')
            if lbl_src.exists():
                if lbl_lnk.exists() or lbl_lnk.is_symlink():
                    lbl_lnk.unlink()
                lbl_lnk.symlink_to(lbl_src.resolve())

            count_this += 1
        count_val_total += count_this
        log.info(f'  ✅ Val lần {i+1} (prefix="{prefix or "gốc"}") → {count_this} symlinks')

    log.info(f'  📦 Tổng: {count_train + count_val_total} ảnh')
    log.info(f'     (train={count_train}, val x{VAL_REPEAT}={count_val_total})')


# ================================================================
# WRITE YAML
# ================================================================

def write_yaml() -> Path:
    yaml_path = WORK_DIR / f'dataset_{RUN_NAME}.yaml'
    combined_img_dir = str(WORK_DIR / 'train_data' / 'images')
    cfg = {
        'train': combined_img_dir,
        'val'  : combined_img_dir,
        'nc'   : 4,
        'names': {0: 'truck', 1: 'car', 2: 'van', 3: 'bus'},
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    log.info(f'dataset.yaml:\n{open(yaml_path).read()}')
    return yaml_path


# ================================================================
# TRAIN
# ================================================================

def train(yaml_path: Path, device):
    from ultralytics import YOLO

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    for cache_f in WORK_DIR.rglob('*.cache'):
        cache_f.unlink()
        log.info(f'  Removed cache: {cache_f}')

    model = YOLO('yolov8n.yaml')
    log.info(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

    model.train(
        data          = str(yaml_path),
        epochs        = EPOCHS,
        imgsz         = IMGSZ,
        batch         = BATCH,
        optimizer     = 'SGD',
        workers       = 4,
        device        = device,
        lr0           = 0.01,
        lrf           = 0.01,
        momentum      = 0.937,
        weight_decay  = 0.0005,
        warmup_epochs = 3,
        close_mosaic  = 20,
        patience      = 0,
        cls_pw        = 1.0,
        mixup         = 0.1,
        hsv_h         = 0.015,
        hsv_s         = 0.7,
        hsv_v         = 0.6,
        fliplr        = 0.5,
        flipud        = 0.0,
        degrees       = 0.0,
        mosaic        = 1.0,
        translate     = 0.1,
        scale         = 0.6,
        cos_lr        = True,
        project       = str(WORK_DIR),
        name          = RUN_NAME,
        pretrained    = False,
        exist_ok      = False,
        verbose       = False,
        seed          = SEED,
        val = False,
    )

    best_pt = WORK_DIR / RUN_NAME / 'weights' / 'best.pt'
    log.info(f'✅ Training xong! Best model: {best_pt}')
    return best_pt


# ================================================================
# MAIN
# ================================================================

def main():
    global log
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    log = logging.getLogger(RUN_NAME)
    log.setLevel(logging.INFO)
    if not log.handlers:
        log.addHandler(logging.StreamHandler(sys.stdout))
        log.addHandler(logging.FileHandler(str(WORK_DIR / f'train_{RUN_NAME}.log')))

    log.info('=' * 60)
    log.info(f'  RUN    : {RUN_NAME}')
    log.info(f'  Labels : {LABEL_DIR}')
    log.info(f'  Mode   : TRAIN + VAL x{VAL_REPEAT} (val gộp {VAL_REPEAT} lần)')
    log.info(f'  Epochs={EPOCHS}  Batch={BATCH}  IMGSZ={IMGSZ}')
    log.info('=' * 60)

    run_dir = WORK_DIR / RUN_NAME
    if run_dir.exists():
        shutil.rmtree(run_dir)
        log.info(f'🗑️  Đã xóa run cũ: {run_dir}')
    else:
        log.info(f'✅ Run dir chưa tồn tại, train fresh')

    device = setup()

    log.info('\n[1/2] Setting up directories...')
    setup_train_dir()

    log.info('\n[2/2] Training...')
    yaml_path = write_yaml()
    best_pt = train(yaml_path, device)

    log.info('\n' + '=' * 60)
    log.info('🏆 Done!')
    log.info(f'   Best model: {best_pt}')
    log.info('=' * 60)


if __name__ == '__main__':
    main()

  RUN    : v11_val5x_200ep
  Labels : /kaggle/input/datasets/cdanht2/cleaned-data-v11/cleaned_data_v11/labels/train
  Mode   : TRAIN + VAL x5 (val gộp 5 lần)
  Epochs=250  Batch=64  IMGSZ=640
✅ Run dir chưa tồn tại, train fresh
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch : 2.10.0+cu128
CUDA    : True
GPUs    : 2
Device  : [0, 1]
  ✅ Train images: /kaggle/input/competitions/3-lc-multi-vehicle-detection-challenge/competition_starter/data/train/images
  ✅ Val images: /kaggle/input/competitions/3-lc-multi-vehicle-detection-challenge/competition_starter/data/val/images
  ✅ Val labels: /kaggle/input/competitions/3-lc-multi-vehicle-detection-challenge/competition_starter/data/val/labels
  ✅ Label dir (train): /kaggle/input